# Exponential smoothing (Holt-Winters) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def make_series(n=60, trend=0.08, amp=2.0, noise=0.25):
    t=np.arange(n)
    return t, 10 + trend*t + amp*np.sin(2*np.pi*t/12) + noise*np.random.randn(n)
def acf(x, max_lag):
    x=np.asarray(x)-np.mean(x); den=np.dot(x,x)
    return np.array([1.0]+[np.dot(x[:-k],x[k:])/den for k in range(1,max_lag+1)])
def moving_average(x,w):
    return np.convolve(x, np.ones(w)/w, mode='valid')
print('setup ok')

## Forecasting by updating level, trend, and season with recency-weighted averages
Exponential smoothing keeps small state variables instead of a whole training set. These examples show simple smoothing, the alpha knob, Holt trend, additive season updates, and a Holt-Winters next-step forecast.

In [ ]:
# 1) simple exponential smoothing with alpha=0.5
y=[10,12,13,12]; alpha=0.5; level=[y[0]]
for obs in y[1:]: level.append(alpha*obs+(1-alpha)*level[-1])
plt.figure(figsize=(6,3)); plt.plot(y,'o-',label='observed'); plt.plot(level,'s-',label='level'); plt.legend(); plt.title('level updates halfway toward each observation')
assert np.allclose(level,[10,11,12,12])

In [ ]:
# 2) alpha controls memory: high alpha reacts faster
y=np.array([10,10,10,14,14,14]); levels=[]
for alpha in [0.2,0.8]:
    l=[y[0]]
    for obs in y[1:]: l.append(alpha*obs+(1-alpha)*l[-1])
    levels.append(l)
plt.figure(figsize=(6,3)); plt.plot(y,'k--',label='observed'); plt.plot(levels[0],label='alpha=.2'); plt.plot(levels[1],label='alpha=.8'); plt.legend(); plt.title('recency weight controls responsiveness')
assert levels[1][3] > levels[0][3]

In [ ]:
# 3) Holt update carries both level and trend
l,b=10,1; y=12; alpha=0.5; beta=0.25; lnew=alpha*y+(1-alpha)*(l+b); bnew=beta*(lnew-l)+(1-beta)*b; forecast=lnew+bnew
plt.figure(figsize=(6,3)); plt.bar(['old level','new level','new trend','forecast'],[l,lnew,bnew,forecast]); plt.title('Holt: level + trend')
assert abs(lnew-11.5)<1e-9 and abs(bnew-1.125)<1e-9 and abs(forecast-12.625)<1e-9

In [ ]:
# 4) additive seasonal state updates the repeated offset
gamma=0.3; observed=15; level_plus_trend=12; old_season=2; new_season=gamma*(observed-level_plus_trend)+(1-gamma)*old_season
plt.figure(figsize=(6,3)); plt.bar(['old season','new evidence','updated season'],[old_season,observed-level_plus_trend,new_season]); plt.title('seasonal offset moves toward residual')
assert abs(new_season-2.3)<1e-9

In [ ]:
# 5) Holt-Winters forecast = level + trend + next seasonal offset
l=20; b=0.5; seasons=np.array([2,-1,-2,1.0]); h=np.arange(1,9); forecasts=l+h*b+np.resize(seasons,8)
plt.figure(figsize=(6,3)); plt.plot(h,forecasts,'o-'); plt.title('seasonal forecast repeats offsets around trend')
assert np.allclose(forecasts[:4],[22.5,20.0,19.5,23.0])